# Support Vector Machine - Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import dotenv

import wandb
from src.api.run import sweep_svm
from src.api.sweep import wandb_sweep

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [3]:
max_runs = 100
sweep_config = {
    "name": "Support Vector Machine",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "svm_config": {
            "parameters": {
                "kernel": {"values": ["poly", "rbf", "sigmoid"]},
                "degree": {"distribution": "int_uniform", "min": 2, "max": 6},
                "gamma": {"values": ["scale", "auto"]},
                "tol": {"distribution": "log_uniform_values", "min": 1e-3 / 2, "max": 2e-3},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "epsilon": {"distribution": "log_uniform_values", "min": 1e-3, "max": 2e-1},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_svm, run_count=max_runs, project="svm")

## Submission from Best Model

In [6]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.svm import SVMHyperparamConfig, SVMRegressorModel
from src.submissions import create_submission

In [8]:
run = wandb.Api().run("bnmsb4qn")
config = run.config
config

{'run_config': {'num_features': 89,
  'start_season': 2003,
  'valid_season': 2024},
 'svm_config': {'C': 0.004791629068058249,
  'tol': 0.000921386933333294,
  'gamma': 'auto',
  'degree': 6,
  'kernel': 'sigmoid',
  'epsilon': 0.0013000418576773437}}

In [9]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=89, valid_season=2024, start_season=2003, data_loader='season_average')

In [10]:
hyperparameters = SVMHyperparamConfig(**config.get("svm_config", {}))
hyperparameters

SVMHyperparamConfig(kernel='sigmoid', degree=6, gamma='auto', coef0=0.0, tol=0.000921386933333294, C=0.004791629068058249, epsilon=0.0013000418576773437, shrinking=True, cache_size=200, verbose=False, max_iter=-1)

In [11]:
model = SVMRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [12]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_svm_{season}.csv", fit=True)

metrics: {'train_brier': np.float64(0.17125833509907898)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_svm_2025.csv')

## Sweeps with Default Features

In [5]:
max_runs = 100
sweep_config = {
    "name": "Support Vector Machine (Default Features)",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "svm_config": {
            "parameters": {
                "kernel": {"values": ["poly", "rbf", "sigmoid"]},
                "degree": {"distribution": "int_uniform", "min": 2, "max": 6},
                "gamma": {"values": ["scale", "auto"]},
                "tol": {"distribution": "log_uniform_values", "min": 1e-3 / 2, "max": 2e-3},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "epsilon": {"distribution": "log_uniform_values", "min": 1e-3, "max": 2e-1},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_svm, run_count=max_runs, project="svm")

### Submission from Best Model

In [7]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.svm import SVMHyperparamConfig, SVMRegressorModel
from src.submissions import create_submission

In [11]:
run = wandb.Api().run("685vhxhd")
config = run.config
config

{'run_config': {'num_features': 0, 'start_season': 2003, 'valid_season': 2024},
 'svm_config': {'C': 0.061695606810578424,
  'tol': 0.0005625439224612987,
  'gamma': 'scale',
  'degree': 2,
  'kernel': 'rbf',
  'epsilon': 0.16231707704003384}}

In [12]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2024, start_season=2003, data_loader='season_average')

In [13]:
hyperparameters = SVMHyperparamConfig(**config.get("svm_config", {}))
hyperparameters

SVMHyperparamConfig(kernel='rbf', degree=2, gamma='scale', coef0=0.0, tol=0.0005625439224612987, C=0.061695606810578424, epsilon=0.16231707704003384, shrinking=True, cache_size=200, verbose=False, max_iter=-1)

In [14]:
model = SVMRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [15]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_svm_default_features_{season}.csv", fit=True)

metrics: {'train_brier': np.float64(0.16245314967507674)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_svm_default_features_2025.csv')

## Sweeps with Weighted Season Average DataLoader

In [3]:
max_runs = 100
sweep_config = {
    "name": "Support Vector Machine",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "svm_config": {
            "parameters": {
                "kernel": {"values": ["poly", "rbf", "sigmoid"]},
                "degree": {"distribution": "int_uniform", "min": 2, "max": 6},
                "gamma": {"values": ["scale", "auto"]},
                "tol": {"distribution": "log_uniform_values", "min": 1e-3 / 2, "max": 2e-3},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "epsilon": {"distribution": "log_uniform_values", "min": 1e-3, "max": 2e-1},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_svm, run_count=max_runs, project="svm")

## Submission from Best Model

In [ ]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.svm import SVMHyperparamConfig, SVMRegressorModel
from src.submissions import create_submission

In [ ]:
run = wandb.Api().run("bnmsb4qn")
config = run.config
config

{'run_config': {'num_features': 89,
  'start_season': 2003,
  'valid_season': 2024},
 'svm_config': {'C': 0.004791629068058249,
  'tol': 0.000921386933333294,
  'gamma': 'auto',
  'degree': 6,
  'kernel': 'sigmoid',
  'epsilon': 0.0013000418576773437}}

In [ ]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=89, valid_season=2024, start_season=2003, data_loader='season_average')

In [ ]:
hyperparameters = SVMHyperparamConfig(**config.get("svm_config", {}))
hyperparameters

SVMHyperparamConfig(kernel='sigmoid', degree=6, gamma='auto', coef0=0.0, tol=0.000921386933333294, C=0.004791629068058249, epsilon=0.0013000418576773437, shrinking=True, cache_size=200, verbose=False, max_iter=-1)

In [ ]:
model = SVMRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [ ]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_svm_{season}.csv", fit=True)

metrics: {'train_brier': np.float64(0.17125833509907898)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_svm_2025.csv')